# The Trinity on GPT-2: Colab runner

Two ways to use this notebook (pick **Runtime → Change runtime type → T4 GPU** first):

* **A. Run here.** The experiments run on the Colab GPU and the figures show inline.
* **B. Colab as a remote GPU.** Colab serves the model through a public tunnel, and you run the experiments
  from your laptop, a cloud session or another notebook with `--remote <URL> --token <TOKEN>`.

In [ ]:
# Setup (both modes)
!git clone -q https://github.com/ignaciourbina/trinity
%cd trinity
!pip install -q -e ".[hf]"

## A. Run the full verification on this runtime

In [ ]:
from trinity import LMConfig, get_backend, run_lm_suite
from trinity.plotting import plot_lm
from trinity.report import print_lm

backend = get_backend("hf", model="gpt2")          # any HF causal LM id works
results = run_lm_suite(backend, LMConfig())        # defaults match the original script
print_lm(results)
plot_lm(results, "outputs", show=True)

## B. Serve the GPU to a remote client

The server runs in a background thread and needs a token on every request.
`cloudflared` opens a free quick tunnel with no account. For a stable URL, use
`tunnel="ngrok"` (run `pip install pyngrok` and set `NGROK_AUTHTOKEN` first).
Keep this runtime open while the client is running.

In [ ]:
from trinity.remote.colab import serve

handle = serve("gpt2", tunnel="cloudflared")       # prints URL, token and the client command

On the client machine (the client needs only numpy and matplotlib, not torch):

```bash
pip install "git+https://github.com/ignaciourbina/trinity"
python -m trinity lm --remote <URL> --token <TOKEN>
```

or from Python:

```python
from trinity import get_backend, run_lm_suite
backend = get_backend("remote", url="<URL>", token="<TOKEN>")
results = run_lm_suite(backend)
```

In [ ]:
# Stop serving when you are done
handle.stop()